In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
dt = 0.01
T = 10.0
t = np.arange(0, T, dt)

# Controller gains
lam = 3.0
k1 = 5.0
k2 = 10.0
eps = 0.01

def sat(x, eps):
    return np.clip(x / eps, -1.0, 1.0)

# Reference trajectory
def x_ref(t):
    return np.sin(t)

def dx_ref(t):
    return np.cos(t)

def ddx_ref(t):
    return -np.sin(t)

# States
x = np.zeros_like(t)
dx = np.zeros_like(t)

# Initial conditions
x[0] = 2.0
dx[0] = 0.0

s_arr = np.zeros_like(t)
u_arr = np.zeros_like(t)

nu = 0.0  # internal Super-Twisting-State

noise_std = 0.2
noise = np.random.normal(0.0, noise_std, size=t.shape)

# Simulation
for i in range(len(t) - 1):
    e = x[i] - x_ref(t[i])
    de = dx[i] - dx_ref(t[i])

    s = de + lam * e
    sigma = sat(s, eps)

    nu = nu - k2 * sigma * dt

    u_sta = -k1 * np.sqrt(abs(s)) * sigma + nu
    u = ddx_ref(t[i]) + noise[i] - lam * de + u_sta

    dx[i + 1] = dx[i] + u * dt
    x[i + 1] = x[i] + dx[i + 1] * dt

    s_arr[i] = s
    u_arr[i] = u


e = x[-1] - x_ref(t[-1])
de = dx[-1] - dx_ref(t[-1])
s_arr[-1] = de + lam * e
    # ---------- Plot ----------
plt.figure()
plt.plot(t, x, label="x")
plt.plot(t, x_ref(t), "--", label="x_ref")
plt.xlabel("Time [s]")
plt.ylabel("Position")
plt.title("Position Tracking")
plt.grid(True)
plt.legend()
plt.show()

plt.figure()
plt.plot(t, x - x_ref(t), label="tracking error e")
plt.xlabel("Time [s]")
plt.ylabel("Error")
plt.title("Tracking Error")
plt.grid(True)
plt.legend()
plt.show()

plt.figure()
plt.plot(t, s_arr, label="sliding variable s")
plt.xlabel("Time [s]")
plt.ylabel("s")
plt.title("Sliding Variable")
plt.grid(True)
plt.legend()
plt.show()

plt.figure()
plt.plot(t, u_arr, label="control input u")
plt.xlabel("Time [s]")
plt.ylabel("u")
plt.title("Control Input")
plt.grid(True)
plt.legend()
plt.show()

# ---------- Phase Plot: Twisting / Super-Twisting behavior ----------
ds_arr = np.gradient(s_arr, dt)
dds_arr = np.gradient(ds_arr, dt)

fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")

ax.plot(s_arr, ds_arr, dds_arr, label="phase trajectory", lw=1.5)
ax.scatter(s_arr[0], ds_arr[0], dds_arr[0], color="green", s=40, label="start")
ax.scatter(s_arr[-1], ds_arr[-1], dds_arr[-1], color="red", s=40, label="end")

ax.set_xlabel("s")
ax.set_ylabel("ds/dt")
ax.set_zlabel("d²s/dt²")
ax.set_title("Phase Plot: Super-Twisting Behavior")
ax.legend()
plt.tight_layout()
plt.show()

# ---------- Zoom near sliding surface ----------
plt.figure()
plt.plot(t, s_arr, label="s")
plt.xlabel("Time [s]")
plt.ylabel("s")
plt.title("Sliding Variable Zoom")
plt.xlim(0, 3)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
import plotly
plotly.offline.init_notebook_mode(connected=True)
fig = go.Figure(data=[go.Scatter3d(x=s_arr, y=ds_arr, z=dds_arr, mode="lines", name="phase trajectory")])
fig.add_trace(go.Scatter3d(x=[s_arr[0]], y=[ds_arr[0]], z=[dds_arr[0]], mode="markers", marker=dict(size=5, color="green"), name="start"))
fig.add_trace(go.Scatter3d(x=[s_arr[-1]], y=[ds_arr[-1]], z=[dds_arr[-1]], mode="markers", marker=dict(size=5, color="red"), name="end"))
fig.update_layout(scene=dict(xaxis_title="s", yaxis_title="ds/dt", zaxis_title="d²s/dt²"), title="Phase Plot: Super-Twisting Behavior")
fig.show()

In [ ]:
from Models.diffdrive import DiffDrive

class SuperTwistingController:
    def __init__(self, lam=3.0, k1=5.0, k2=10.0, eps=0.01):
        self.lam = lam
        self.k1 = k1
        self.k2 = k2
        self.eps = eps
        self.nu = 0.0  # internal Super-Twisting-State
        self.prev_state = np.zeros(3)  # [x, y, phi]
        self.prev_waypoint = np.zeros(2)  # [x_ref, y_ref]
        self.v_cmd = 0.5  # desired linear velocity
        self.L = 0.5  # wheelbase
        self.r = 0.2  # wheel radius

    def sat(self, x):
        return np.clip(x / self.eps, -1.0, 1.0)
    
    def wrap_angle(self, angle):
        return (angle + np.pi) % (2 * np.pi) - np.pi

    def compute_control(self, x, y, phi, waypoint, dt):
        
        phi_des = np.arctan2(waypoint[1] - y, waypoint[0] - x)
        s = self.wrap_angle(phi - phi_des)

        sigma = self.sat(s)

        self.nu -= self.k2 * sigma * dt

        u_sta = -self.k1 * np.sqrt(abs(s)) * sigma + self.nu
        #u = self.prev_state[2] - self.lam * de + u_sta
        omega_l = (self.v_cmd - (self.L / 2) * u_sta) / self.r
        omega_r = (self.v_cmd + (self.L / 2) * u_sta) / self.r
        #e_prev = e
        #de = (phi - self.prev_state[2]) / dt
        #self.prev_state = np.array([x, y, phi]) / dt
        #self.prev_waypoint = waypoint

        return omega_l, omega_r, s, self.nu, u_sta

In [ ]:
sim_t = 10.0
dt = 0.01
t = np.arange(0, sim_t, dt)

zoh = 0.4
waypoint = []  # [x_ref, y_ref]
time = 0.0

while time < sim_t:

    waypoint.append([time, np.sin(time)])
    time += zoh

waypoints = np.array(waypoint)

plt.figure()
plt.scatter(waypoints[:, 0], waypoints[:, 1], label="reference trajectory", color="turquoise", s=10)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Reference Trajectory")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
robot = DiffDrive()
st_controller = SuperTwistingController()

i = 0
dt = 0.01
time = 0.0

states = []
time_arr = []

omega_l_arr = []
omega_r_arr = []
u_sta_arr = []

s_arr = []
nu_arr = []
waypoint_index_arr = []

while i < len(waypoints):
    omega_l, omega_r, s, nu, u_sta = st_controller.compute_control(robot.x, robot.y, robot.phi, waypoints[i], dt)
    new_state = robot.update(omega_l, omega_r, dt)

    new_state = robot.update(omega_l, omega_r, dt)

    states.append(new_state)
    time_arr.append(time)

    omega_l_arr.append(omega_l)
    omega_r_arr.append(omega_r)
    u_sta_arr.append(u_sta)

    s_arr.append(s)
    nu_arr.append(nu)
    waypoint_index_arr.append(i)
    if np.linalg.norm([robot.x - waypoints[i][0], robot.y - waypoints[i][1]]) < 0.1:
        print(f"Reached waypoint {i} at time {t[i]:.2f}s")
        i += 1

    time += dt

states = np.array(states)
time_arr = np.array(time_arr)

omega_l_arr = np.array(omega_l_arr)
omega_r_arr = np.array(omega_r_arr)
u_sta_arr = np.array(u_sta_arr)

s_arr = np.array(s_arr)
nu_arr = np.array(nu_arr)
waypoint_index_arr = np.array(waypoint_index_arr)

plt.figure(figsize=(7, 6))
plt.scatter(states[:, 0], states[:, 1], color="orange", label="robot trajectory", s=1)
plt.scatter(waypoints[:, 0], waypoints[:, 1], color="blue", s=25, label="waypoints")

plt.xlabel("x in meters")
plt.ylabel("y in meters")
plt.title("Robot Trajectory")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_arr, s_arr, color="purple", label="s")

plt.xlabel("time in seconds")
plt.ylabel("s in radians")
plt.title("Sliding Variable")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_arr, nu_arr, color="green", label="nu")

plt.xlabel("time in seconds")
plt.ylabel("nu")
plt.title("Internal Super-Twisting Integrator State")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_arr, omega_l_arr, label="omega_l", color="tab:blue")
plt.plot(time_arr, omega_r_arr, label="omega_r", color="tab:orange")

plt.xlabel("time in seconds")
plt.ylabel("wheel angular velocity in rad/s")
plt.title("Wheel Velocities")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(6, 5))
plt.plot(s_arr, nu_arr, color="black")

plt.xlabel("s")
plt.ylabel("nu")
plt.title("Super-Twisting Phase Plot: s-nu")
plt.grid(True)
plt.show()

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")

ax.plot(s_arr, nu_arr, u_sta_arr, color="purple")

ax.set_xlabel("s")
ax.set_ylabel("nu")
ax.set_zlabel("u_sta")
ax.set_title("Super-Twisting Behaviour in s-nu-u Space")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class DiffDrive:
    def __init__(self):
        self.r = 0.2
        self.L = 0.5

        self.x = 0.0
        self.y = 0.0
        self.phi = 0.0
        self.v = 0.0
        self.omega = 0.0

    def update(self, omega_l, omega_r, dt):
        self.v = self.r / 2.0 * (omega_r + omega_l)
        self.omega = self.r / self.L * (omega_r - omega_l)

        self.x += self.v * np.cos(self.phi) * dt
        self.y += self.v * np.sin(self.phi) * dt
        self.phi = wrap_to_pi(self.phi + self.omega * dt)

        return np.array([self.x, self.y, self.phi, self.v, self.omega])


def wrap_to_pi(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi


def sat(x):
    return np.clip(x, -1.0, 1.0)


class SuperTwistingPoseController:
    def __init__(
        self,
        k1_phi=4.0,
        k2_phi=2.0,
        k1_v=2.0,
        k2_v=1.0,
        v_ref=0.3
    ):
        self.k1_phi = k1_phi
        self.k2_phi = k2_phi
        self.k1_v = k1_v
        self.k2_v = k2_v

        self.v_ref = v_ref

        self.r = 0.2
        self.L = 0.5

        self.nu_phi = 0.0
        self.nu_v = 0.0

        self.eps_phi = 0.05
        self.eps_v = 0.05

        self.nu_phi_max = 8.0
        self.nu_v_max = 2.0

        self.omega_max = 10.0
        self.v_max = 1.0
        self.v_min = 0.0

    def compute_control(self, x, y, phi, v, waypoint, dt):
        dx = waypoint[0] - x
        dy = waypoint[1] - y
        dist = np.hypot(dx, dy)

        phi_des = np.arctan2(dy, dx)

        s_phi = wrap_to_pi(phi - phi_des)
        sigma_phi = sat(s_phi / self.eps_phi)

        self.nu_phi += -self.k2_phi * sigma_phi * dt
        self.nu_phi = np.clip(self.nu_phi, -self.nu_phi_max, self.nu_phi_max)

        omega_cmd = -self.k1_phi * np.sqrt(abs(s_phi)) * sigma_phi + self.nu_phi
        omega_cmd = np.clip(omega_cmd, -self.omega_max, self.omega_max)

        v_des = min(self.v_ref, 0.8 * dist)

        s_v = v - v_des
        sigma_v = sat(s_v / self.eps_v)

        self.nu_v += -self.k2_v * sigma_v * dt
        self.nu_v = np.clip(self.nu_v, -self.nu_v_max, self.nu_v_max)

        a_cmd = -self.k1_v * np.sqrt(abs(s_v)) * sigma_v + self.nu_v

        v_cmd = v + a_cmd * dt
        v_cmd = np.clip(v_cmd, self.v_min, self.v_max)

        omega_l = (v_cmd - (self.L / 2.0) * omega_cmd) / self.r
        omega_r = (v_cmd + (self.L / 2.0) * omega_cmd) / self.r

        return (
            omega_l,
            omega_r,
            v_cmd,
            omega_cmd,
            s_phi,
            s_v,
            self.nu_phi,
            self.nu_v,
            phi_des,
            v_des,
            dist
        )
    

robot = DiffDrive()
controller = SuperTwistingPoseController(
    k1_phi=4.0,
    k2_phi=2.0,
    k1_v=2.0,
    k2_v=1.0,
    v_ref=0.3
)

dt = 0.01
time = 0.0
i = 0

states = []
time_arr = []

omega_l_arr = []
omega_r_arr = []
omega_cmd_arr = []
v_cmd_arr = []

s_phi_arr = []
s_v_arr = []
nu_phi_arr = []
nu_v_arr = []

v_des_arr = []
phi_des_arr = []
dist_arr = []
waypoint_index_arr = []

while i < len(waypoints):
    omega_l, omega_r, v_cmd, omega_cmd, s_phi, s_v, nu_phi, nu_v, phi_des, v_des, dist = (
        controller.compute_control(
            robot.x,
            robot.y,
            robot.phi,
            robot.v,
            waypoints[i],
            dt
        )
    )

    new_state = robot.update(omega_l, omega_r, dt)

    states.append(new_state)
    time_arr.append(time)

    omega_l_arr.append(omega_l)
    omega_r_arr.append(omega_r)
    omega_cmd_arr.append(omega_cmd)
    v_cmd_arr.append(v_cmd)

    s_phi_arr.append(s_phi)
    s_v_arr.append(s_v)
    nu_phi_arr.append(nu_phi)
    nu_v_arr.append(nu_v)

    v_des_arr.append(v_des)
    phi_des_arr.append(phi_des)
    dist_arr.append(dist)
    waypoint_index_arr.append(i)

    if dist < 0.1:
        print(f"Reached waypoint {i} at time {time:.2f}s")
        i += 1

    time += dt 


states = np.array(states)
time_arr = np.array(time_arr)

omega_l_arr = np.array(omega_l_arr)
omega_r_arr = np.array(omega_r_arr)
omega_cmd_arr = np.array(omega_cmd_arr)
v_cmd_arr = np.array(v_cmd_arr)

s_phi_arr = np.array(s_phi_arr)
s_v_arr = np.array(s_v_arr)
nu_phi_arr = np.array(nu_phi_arr)
nu_v_arr = np.array(nu_v_arr)

v_des_arr = np.array(v_des_arr)
phi_des_arr = np.array(phi_des_arr)
dist_arr = np.array(dist_arr)
waypoint_index_arr = np.array(waypoint_index_arr)

x_arr = states[:, 0]
y_arr = states[:, 1]
phi_arr = states[:, 2]
v_arr = states[:, 3]
omega_arr = states[:, 4]

plt.figure(figsize=(7, 6))
plt.plot(x_arr, y_arr, color="orange", label="robot trajectory")
plt.scatter(waypoints[:, 0], waypoints[:, 1], color="blue", s=25, label="waypoints")

plt.xlabel("x in meters")
plt.ylabel("y in meters")
plt.title("Robot Trajectory with Heading and Velocity Control")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_arr, v_arr, label="v actual", color="tab:blue")
plt.plot(time_arr, v_des_arr, label="v desired", color="tab:green", linestyle="--")
plt.plot(time_arr, v_cmd_arr, label="v command", color="tab:red", alpha=0.8)

plt.xlabel("time in seconds")
plt.ylabel("velocity in m/s")
plt.title("Velocity Tracking")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_arr, s_v_arr, color="purple", label="s_v = v - v_des")

plt.xlabel("time in seconds")
plt.ylabel("s_v in m/s")
plt.title("Velocity Sliding Variable")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_arr, nu_v_arr, color="green", label="nu_v")

plt.xlabel("time in seconds")
plt.ylabel("nu_v")
plt.title("Velocity Super-Twisting Integrator State")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_arr, s_phi_arr, color="tab:orange", label="s_phi")

plt.xlabel("time in seconds")
plt.ylabel("s_phi in radians")
plt.title("Heading Sliding Variable")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_arr, omega_arr, label="omega actual", color="tab:blue")
plt.plot(time_arr, omega_cmd_arr, label="omega command", color="tab:red", alpha=0.8)

plt.xlabel("time in seconds")
plt.ylabel("angular velocity in rad/s")
plt.title("Angular Velocity")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(time_arr, omega_l_arr, label="omega_l", color="tab:blue")
plt.plot(time_arr, omega_r_arr, label="omega_r", color="tab:orange")

plt.xlabel("time in seconds")
plt.ylabel("wheel angular velocity in rad/s")
plt.title("Wheel Velocities")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(6, 5))
plt.plot(s_v_arr, nu_v_arr, color="black")

plt.xlabel("s_v")
plt.ylabel("nu_v")
plt.title("Velocity Super-Twisting Phase Plot: s_v-nu_v")
plt.grid(True)
plt.show()

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection="3d")

ax.plot(s_v_arr, nu_v_arr, v_cmd_arr, color="purple")

ax.set_xlabel("s_v")
ax.set_ylabel("nu_v")
ax.set_zlabel("v_cmd")
ax.set_title("Velocity Super-Twisting Behaviour in s_v-nu-v Space")

plt.show()